In [ ]:
import os
import cv2
import glob
import shutil
from tqdm import tqdm

# --- CONFIGURATION ---

SOURCE_DIR = "UCF-101" 
TEMP_DIR = "temp_jpg_224"     
ZIP_NAME = "UCF101_Images_224" 

RESIZE_DIM = 224
FRAMES_PER_CLIP = 16

# --- SAFETY CHECK ---
if not os.path.exists(SOURCE_DIR):
    print(f"ERROR: Cannot find folder '{SOURCE_DIR}'!")
    print("   Make sure the path is correct or move the script near the folder.")
    exit()

# --- START CONVERSION ---
print(f"Starting conversion VIDEO -> JPG (Resize: {RESIZE_DIM}x{RESIZE_DIM})...")

if os.path.exists(TEMP_DIR):
    print("   Cleaning up old temporary folder...")
    shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)

# Find videos (searches for .avi files)
video_paths = glob.glob(os.path.join(SOURCE_DIR, "*", "*.avi"))
print(f"   Found {len(video_paths)} videos.")

for video_path in tqdm(video_paths, desc="Processing"):
    try:
        class_name = os.path.basename(os.path.dirname(video_path))
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        
        save_folder = os.path.join(TEMP_DIR, class_name, video_name)
        os.makedirs(save_folder, exist_ok=True)
        
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            # Corrupted or empty video
            continue

        # Calculate frame indices
        if total_frames > FRAMES_PER_CLIP:
            step = total_frames / FRAMES_PER_CLIP
            indices = [int(i * step) for i in range(FRAMES_PER_CLIP)]
        else:
            indices = list(range(total_frames))
                
        frame_idx = 0
        saved_count = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            if frame_idx in indices and saved_count < FRAMES_PER_CLIP:
                frame = cv2.resize(frame, (RESIZE_DIM, RESIZE_DIM))
                cv2.imwrite(os.path.join(save_folder, f"frame_{saved_count:02d}.jpg"), frame)
                saved_count += 1
            frame_idx += 1
        cap.release()
        
        # Padding (if the video is too short, duplicate the last frame)
        if saved_count < FRAMES_PER_CLIP and saved_count > 0:
            last_frame_path = os.path.join(save_folder, f"frame_{saved_count-1:02d}.jpg")
            if os.path.exists(last_frame_path):
                img = cv2.imread(last_frame_path)
                for i in range(saved_count, FRAMES_PER_CLIP):
                    cv2.imwrite(os.path.join(save_folder, f"frame_{i:02d}.jpg"), img)
                    
    except Exception as e:
        print(f"\nError on {video_name}: {e}")
        continue

# --- CREATE ZIP ---
print("\Creating ZIP archive (this may take a few minutes)...")
shutil.make_archive(ZIP_NAME, 'zip', TEMP_DIR)

print(f"\DONE!")